# Step 3: Model Evaluation


Evaluate the performance of the embedding-based skill mapping system using LLM to annotate ground truth labels.


- **Method**: Hybrid embedding method (fuzzy + semantic similarity)
- **Ground truth**: LLM-validated ESCO mappings
- **Evaluation setup**: For each extracted skill, the system suggests top-3 ESCO matches. We evaluate if the correct answer appears in these suggestions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import brier_score_loss
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
np.random.seed(42)

## Load Data

In [ ]:
# Load the data
df = pd.read_csv('embedding_method_with_ground_truth.csv')
print(f"Total items: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## Data Overview

In [ ]:
# Ground truth distribution
print("Ground truth label distribution:")
print("="*80)
gt_counts = df['ground_truth_esco'].value_counts()
print(gt_counts.head(20))

# Categorize items
valid_items = df[~df['ground_truth_esco'].isin(['no_match', 'invalid_extraction'])]
no_match_items = df[df['ground_truth_esco'] == 'no_match']
invalid_items = df[df['ground_truth_esco'] == 'invalid_extraction']

print(f"\n{'='*80}")
print(f"Valid items (has ESCO match): {len(valid_items)} ({len(valid_items)/len(df)*100:.1f}%)")
print(f"No match (ESCO coverage gap): {len(no_match_items)} ({len(no_match_items)/len(df)*100:.1f}%)")
print(f"Invalid extraction: {len(invalid_items)} ({len(invalid_items)/len(df)*100:.1f}%)")

### Key Observations

- Only **30% of items** have valid ESCO ground truth labels — the majority are either taxonomy misses (`no_match`) or invalid extractions.
- The `invalid_extraction` rate (15.2%) suggests the extraction step is tagging job logistics (e.g., contract terms, location) rather than transferable skills.
- The high `no_match` rate (54.9%) can be due to **ESCO coverage gaps** for emerging or highly specific technical skills or mapping method limitations.
- Evaluation is conducted on the **83 valid items only** — the meaningful subset where model performance can be fairly assessed.

In [ ]:
# Item type distribution
print("\nDistribution by item type:")
print(df['item_type'].value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground truth categories
categories = ['Valid ESCO Match', 'No Match', 'Invalid Extraction']
counts = [len(valid_items), len(no_match_items), len(invalid_items)]
colors = ['#2ecc71', '#e74c3c', '#95a5a6']
axes[0].bar(categories, counts, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Ground Truth Categories')
axes[0].set_ylim([0, max(counts) * 1.1])
for i, v in enumerate(counts):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=11, fontweight='bold')

# Item types
item_type_counts = df['item_type'].value_counts()
axes[1].bar(range(len(item_type_counts)), item_type_counts.values, alpha=0.7, edgecolor='black')
axes[1].set_xticks(range(len(item_type_counts)))
axes[1].set_xticklabels(item_type_counts.index)
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution by Item Type')

plt.tight_layout()
plt.show()

In [ ]:
# Filter to items with valid matches
items_with_matches = df[~df['ground_truth_esco'].isin(['no_match', 'invalid_extraction'])].copy()

print(f"Items with valid ESCO matches: {len(items_with_matches)}")
print(f"\nBreakdown by item type:")
print(items_with_matches['item_type'].value_counts())

---
## Metric Selection and Justification

### Chosen Metric: Precision@3

**Definition:** For each extracted skill/responsibility/requirement, the system returns up to 3 ESCO suggestions. Precision@3 measures the fraction of those 3 suggestions that are correct (i.e., appear in the ground truth ESCO label set).

For each item with valid ground truth:

$$P@3 = \frac{\text{number of relevant predictions in top-3}}{3}$$


Example:
- Ground truth: "SQL;MySQL" and only 1 prediction matches -> P@3 = 1/3 = 0.333

### Alternative Metrics Considered

**1. Micro-F1 (for exact matches)**
- Captures overall precision/recall balance
- Treats prediction as unordered set
- Does not account for ranking quality
- Less aligned with shortlist-based UI interaction

**2. Hierarchical F1 (for partial credit)**
- ESCO has a 4-level taxonomy hierarchy
- Could reward parent/child matches (e.g., “SQL” vs “MySQL”)
- Current system performs flat matching
- Incorporating hierarchy is a valid future improvement

**3. Mean Reciprocal Rank (MRR)**
- Strongly rewards correct prediction at rank 1
- May over-penalize useful suggestions appearing at rank 2 or 3
- Less suitable when multiple relevant labels may exist

### Why Precision@3 Over Other Metrics?


Precision@3 captures two critical trade-offs for this use case:

**1. Over-tagging risk:** If the system returns 3 ESCO labels but only 1 is correct, the advertiser sees misleading tag suggestions. Precision@3 penalises this directly — a score of 1/3 reflects the noise. In a recruiter-facing product, over-tagging clutters ads with irrelevant skills and harms candidate matching quality.

**2. Missing critical skills:** Missing a key skill (e.g. labelling a DevOps role without "CI/CD") reduces candidate match quality. Precision@3 gives a holistic signal about how reliably the system surfaces correct labels within its top-3 retrieval window, simulating a user-selects-from-suggestions UX.

**3. Why top-3 specifically?** In a realistic advertiser-facing UI, presenting more than 3 suggestions creates cognitive overload. Top-3 is the practical retrieval budget, making Precision@3 the metric most aligned with product constraints.


---
## 4. Precision@3 Point Estimate

In [ ]:
def calculate_precision_at_k(df, k=3):
    """
    Calculate Precision@k for each item.
    
    P@k = (number of relevant predictions in top-k) / k
    
    Where 'relevant' means the prediction appears in the ground truth labels.
    
    Args:
        df: DataFrame with ground_truth_esco and esco_match_1/2/3 columns
        k: Number of top predictions to consider (default: 3)
    
    Returns:
        DataFrame with precision scores and match details
    """
    results = []
    
    for idx, row in df.iterrows():
        # Parse ground truth (may contain multiple labels)
        gt_labels = set(label.strip() for label in row['ground_truth_esco'].split(';'))
        
        # Get predictions
        predictions = []
        for i in range(1, k+1):
            pred = row[f'esco_match_{i}'] if pd.notna(row.get(f'esco_match_{i}')) else None
            predictions.append(pred)
        
        # Count how many predictions are relevant
        relevant_count = sum(1 for pred in predictions if pred and pred in gt_labels)
        
        # Calculate precision@k
        precision_at_k = relevant_count / k
        
        # Also track which predictions matched
        matches = [pred in gt_labels if pred else False for pred in predictions]
        
        results.append({
            'idx': idx,
            'extracted_text': row['extracted_text'],
            'item_type': row['item_type'],
            'ground_truth': row['ground_truth_esco'],
            'num_gt_labels': len(gt_labels),
            'pred_1': predictions[0],
            'pred_2': predictions[1] if len(predictions) > 1 else None,
            'pred_3': predictions[2] if len(predictions) > 2 else None,
            'match_1': matches[0],
            'match_2': matches[1] if len(matches) > 1 else False,
            'match_3': matches[2] if len(matches) > 2 else False,
            'relevant_count': relevant_count,
            f'precision_at_{k}': precision_at_k,
            'confidence_1': row['confidence_1'],
            'confidence_2': row['confidence_2'],
            'confidence_3': row['confidence_3'],
        })
    
    return pd.DataFrame(results)

# Calculate Precision@3 for all items
eval_df = calculate_precision_at_k(items_with_matches, k=3)

# Calculate mean Precision@3
mean_p_at_3 = eval_df['precision_at_3'].mean()

print("Precision@3 Metric:")
print(f"  Mean P@3: {mean_p_at_3:.3f}")

print(f"\nP@3 Distribution:")
print(f"  Perfect score (P@3 = 1.0): {(eval_df['precision_at_3'] == 1.0).sum()} ({(eval_df['precision_at_3'] == 1.0).mean()*100:.1f}%)")
print(f"  Partial match (0 < P@3 < 1): {((eval_df['precision_at_3'] > 0) & (eval_df['precision_at_3'] < 1)).sum()} ({((eval_df['precision_at_3'] > 0) & (eval_df['precision_at_3'] < 1)).mean()*100:.1f}%)")

print(f"\nGround truth statistics:")
print(f"  Items with single GT label: {(eval_df['num_gt_labels'] == 1).sum()}")
print(f"  Items with multiple GT labels: {(eval_df['num_gt_labels'] > 1).sum()}")

In [ ]:
# Plot P@3 Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of P@3 scores
ax = axes[0]
ax.hist(eval_df['precision_at_3'], bins=20, color='#3498db', alpha=0.7, edgecolor='black')
ax.set_xlabel('Precision@3', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Precision@3 Scores', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Count of perfect/partial
ax = axes[1]
categories = ['Perfect\n(P@3=1.0)', 'Partial\n(0<P@3<1)']
counts = [
    (eval_df['precision_at_3'] == 1.0).sum(),
    ((eval_df['precision_at_3'] > 0) & (eval_df['precision_at_3'] < 1)).sum()
]
colors = ['#2ecc71', '#f39c12']
ax.bar(categories, counts, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Count', fontsize=12)
ax.set_title('P@3 Performance Categories', fontsize=14, fontweight='bold')
for i, (cat, count) in enumerate(zip(categories, counts)):
    pct = count / len(eval_df) * 100
    ax.text(i, count + 2, f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Precision by Item Type

In [ ]:
# Calculate P@3 by item type
type_metrics = eval_df.groupby('item_type').agg({
    'precision_at_3': ['mean', 'count'],
}).round(3)

print("Precision@3 by Item Type:")
print(type_metrics)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
type_p3 = eval_df.groupby('item_type')['precision_at_3'].mean().sort_values(ascending=False)
type_p3.plot(kind='bar', ax=ax, color='#3498db', alpha=0.7, edgecolor='black')
ax.set_xlabel('Item Type', fontsize=12)
ax.set_ylabel('Mean Precision@3', fontsize=12)
ax.set_title('Precision@3 by Item Type', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 95% Bootstrap Confidence Interval

We use bootstrapping to quantify uncertainty in our Precision@3 estimate.

**Method:**
1. Resample the valid items with replacement (1000 iterations)
2. Calculate Precision@3 on each resample
3. Compute 2.5th and 97.5th percentiles -> 95% CI

In [ ]:
## Bootstrap Confidence Intervals
def compute_precision_at_3(y_true, y_pred_top3, k=3):
    """Compute Precision@k for lists of ground truth labels and predictions."""
    scores = []
    for gt, preds in zip(y_true, y_pred_top3):
        if isinstance(gt, str):
            gt_labels = set(label.strip() for label in gt.split(';') if label.strip())
        else:
            gt_labels = set(gt) if gt is not None else set()
        preds = [p for p in (preds or []) if p and pd.notna(p)]
        relevant = sum(1 for p in preds[:k] if p in gt_labels)
        scores.append(relevant / k)
    return float(np.mean(scores)) if len(scores) > 0 else 0.0

def bootstrap_metric(y_true, y_pred_top3, metric_func, n_iterations=1000, confidence=0.95, random_state=None):
    """
    Compute bootstrap point estimate and percentile confidence interval for a metric.

    Returns: point_estimate, lower_bound, upper_bound, bootstrap_scores
    """
    # Normalize inputs to lists
    y_true = list(y_true)
    y_pred_top3 = list(y_pred_top3)
    n_samples = len(y_true)
    if n_samples == 0:
        raise ValueError('Empty input to bootstrap_metric')

    rng = np.random.default_rng(random_state)
    bootstrap_scores = np.empty(n_iterations)

    for i in range(n_iterations):
        # Draw sample indices with replacement
        indices = rng.integers(0, n_samples, n_samples)
        y_true_boot = [y_true[idx] for idx in indices]
        y_pred_boot = [y_pred_top3[idx] for idx in indices]
        bootstrap_scores[i] = metric_func(y_true_boot, y_pred_boot)

    alpha = 1 - confidence
    lower_bound = np.percentile(bootstrap_scores, 100 * (alpha / 2))
    upper_bound = np.percentile(bootstrap_scores, 100 * (1 - alpha / 2))
    point_estimate = float(np.mean(bootstrap_scores))
    return point_estimate, lower_bound, upper_bound, bootstrap_scores

# Prepare inputs from `eval_df` (created earlier by calculate_precision_at_k)
y_true = eval_df['ground_truth'].tolist()
y_pred_top3 = eval_df[['pred_1', 'pred_2', 'pred_3']].apply(lambda r: [r['pred_1'], r['pred_2'], r['pred_3']], axis=1).tolist()

# Run bootstrap (this may take a moment depending on n_iterations)
point_est, ci_lower, ci_upper, bootstrap_scores = bootstrap_metric(
    y_true,
    y_pred_top3,
    compute_precision_at_3,
    n_iterations=1000,
    confidence=0.95,
    random_state=42
)

print(f"\nPrecision@3 with 95% Bootstrap CI:")
print(f"  Point Estimate: {point_est:.3f}")
print(f"  95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"  CI Width: {ci_upper - ci_lower:.3f}")

# Visualize bootstrap distribution
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_scores, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(point_est, color='red', linestyle='--', label='Point Estimate')
plt.axvline(ci_lower, color='green', linestyle='--', label='95% CI Lower')
plt.axvline(ci_upper, color='green', linestyle='--', label='95% CI Upper')
plt.xlabel('Precision@3')
plt.ylabel('Frequency')
plt.title('Bootstrap Distribution of Precision@3')
plt.legend()
plt.show()

## Calibration Analysis

Although confidence scores are available for the top-k retrieved skills, the evaluation operates at the set level using Precision@3 and does not construct prediction-level confidence–correctness pairs. Calibration metrics such as Expected Calibration Error require well-defined probabilistic predictions for each possible class, which is not available in this setup. Furthermore, the Brier Score is also not applicable because it requires well-defined probabilistic predictions for each possible label (i.e., a probability distribution over the full skill space), whereas the model only produces ranked similarity scores for a limited top-k subset. Therefore, neither ECE nor Brier can be meaningfully computed in this evaluation setting.

## Summary of Results

In [ ]:
no_match_count = int(df['ground_truth_esco'].eq('no_match').sum())
invalid_count = int(df['ground_truth_esco'].eq('invalid_extraction').sum())

print("="*80)
print("EVALUATION SUMMARY")
print("="*80)

print(f"\nDataset Statistics:")
print(f"  Total items in dataset: {len(df)}")
print(f"  Items with valid ESCO matches: {len(items_with_matches)} ({len(items_with_matches)/len(df)*100:.1f}%)")
# Ensure no_match_count / invalid_count are defined (derived from earlier filters)
no_match_count = len(no_match_items) if 'no_match_items' in globals() else df['ground_truth_esco'].eq('no_match').sum()
invalid_count = len(invalid_items) if 'invalid_items' in globals() else df['ground_truth_esco'].eq('invalid_extraction').sum()
print(f"  Items marked as 'no_match': {no_match_count} ({no_match_count/len(df)*100:.1f}%)")
print(f"  Items marked as 'invalid_extraction': {invalid_count} ({invalid_count/len(df)*100:.1f}%)")

print(f"\nCore Metric - Precision@3 (on {len(eval_df)} valid items):")
print(f"  Point Estimate:   {mean_p_at_3:.3f}")
print(f"  95% Bootstrap CI: [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"  CI Width:         {ci_upper - ci_lower:.3f}")

print("\n" + "="*80)

### Interpretation

- A **Precision@3 of 0.530** means that on average, roughly **53% of the 3 ESCO suggestions** returned for each valid item are correct. For a baseline hybrid embedding system with no domain fine-tuning, this is a reasonable starting point.
- The **CI of [0.478, 0.582]** reflects moderate uncertainty, partly due to the small evaluation set (n=83). Results should be interpreted cautiously.
- The near-symmetric bootstrap distribution suggests stable performance — the metric is not dominated by a small number of outlier items.

## Assumptions & Limitations

### Data Quirks

1. **Small evaluation set (n=83 valid items):** Only 30% of extracted items had valid ESCO ground truth. Results may not generalise across all industries, seniority levels, or job types in the SEEK corpus.
2. **LLM as annotator:** Ground truth labels were created via LLM zero-shot annotation, not by human domain experts. LLM annotation may contain hallucinations, inconsistent label granularity, or systematic biases aligned with the LLM training data.
3. **Stratified sample may misrepresent corpus:** 100 ads from a stratified sample do not guarantee proportional representation of all industry verticals (e.g., healthcare, trades, finance).
4. **High invalid extraction rate (15.2%):** Many extracted requirements are job logistics (contract length, location, salary) — not transferable skills. This inflates the denominator in any full-corpus evaluation.

### Taxonomy Coverage Gaps

1. **Granularity mismatch:** ESCO uses general terms (e.g., "cloud computing") while job ads reference specific implementations ("AWS Lambda", "Azure Kubernetes Service"). This causes legitimate misses.
2. **Emerging skills not in ESCO:** Rapidly evolving areas like `ci/cd`, `react`, `LLM fine-tuning`, and `prompt engineering` have no direct ESCO equivalent, inflating the `no_match` rate (54.9%).
3. **Language scope:** This evaluation covers English-only postings. Performance in multilingual settings is untested.

### Modeling Shortcuts

1. **Generic embeddings:** The hybrid system uses a general-purpose sentence transformer not fine-tuned on job ad or ESCO data.
2. **Flat matching (no hierarchy):** ESCO is hierarchical, but evaluation uses exact string matching. Parent-child relationships (e.g., SQL vs MySQL) receive no partial credit, likely underestimating semantic performance.
3. **No context utilisation:** Only the extracted skill text is embedded — the surrounding job description context is not leveraged.
4. **Single annotator for ground truth:** LLM ground truth was not cross-validated against multiple annotators or human experts, so inter-annotator reliability is unknown.

## Alternatives & Risks

### Comparison with Alternative Approaches

| Approach | Strengths | Weaknesses |
|---|---|---|
| **Rule-based / keyword matching** | Fully interpretable, zero inference cost, deterministic | Cannot handle synonyms or paraphrasing, requires manual maintenance, fails on unseen skill names |
| **Zero-shot LLM classification** | Handles unseen skills, understands context and intent | High latency (~seconds/item); high API cost at scale; non-deterministic; prompt-sensitive; difficult to audit |
| **Fuzzy + embedding (current approach)** | Low latency, stable and reproducible | No contextual understanding, generic embeddings underperform on domain-specific language, limited to taxonomy coverage |
| **Fine-tuned bi-encoder (recommended)** | Domain-adapted, handles synonymy and context, still fast at inference | Requires labelled training data, periodic update cycle needed |

The current hybrid approach offers the best balance of **latency, cost, and consistency**. A fine-tuned encoder represents the optimal long-term path.


---

### Key Risks

1. **Embedding bias:** Pre-trained transformers over-represent dominant industries and job markets (typically US/UK tech). Niche Australian sectors (mining, agriculture) may be systematically under-served.
2. **Taxonomy drift:** ESCO updates periodically. Cached embeddings and mappings must be refreshed with each taxonomy version, requiring an automated re-indexing pipeline.
3. **Model drift:** Job market language evolves over time. Embedding model performance degrades without periodic re-evaluation and retraining.
4. **Extraction quality dependency:** The mapping system quality is bounded by the extraction step. AI-generated or templated job ads may cause extraction failures that propagate to bad mappings.
5. **Over-tagging in production:** Without a confidence threshold, the system may surface irrelevant ESCO tags that confuse advertisers or harm downstream candidate matching.


## Product & Scaling Considerations

### Helping Advertisers Write Clearer, More Concise Ads

- **Standardised skill tagging UI:** Present the top-3 ESCO suggestions inline as the advertiser types, allowing one-click confirmation. This nudges advertisers toward standardised terminology without mandating it.
- **Ambiguity detection:** Items flagged as `no_match` or with low confidence can trigger a prompt like "We could not map this skill - could you be more specific?", improving ad quality at the source.
- **Reduces redundancy:** Mapping to ESCO skill names, discouraging both vague descriptions and overly specific jargon.
- **Structured analytics downstream:** Standardised ESCO tags enable cross-ad skill frequency analysis, helping SEEK identify market-wide skill demand trends.

### Near-Real-Time Ingestion of New Postings

- **Embedding generation is fast:** Encoding a job ad extracted skills takes with low latency, making synchronous per-ad processing at ingestion time feasible.
- **No generative inference per item:** Unlike LLM-based methods, no expensive decoding is required, keeping latency within acceptable limits.

### Scaling to Tens of Millions of Ads Across Regions and Languages

- **Horizontal scaling:** Embedding inference scales easily and FAISS or managed vector databases can handle millions of vectors.
- **Multilingual support:** Switching to multilingual embeddings (mBERT, XLM-R) allows the same pipeline to support non-English ads with minimal architectural changes.
- **Regional taxonomy:** ESCO supports 27 EU languages natively. For other regions (US, APAC), O*NET or local taxonomies can be indexed alongside ESCO and served via the same retrieval interface.
- **ESCO stability:** The ESCO taxonomy is updated infrequently (~yearly).

## Recommended improvements

### Short-Term
1. **Improve extraction quality:** Add a classifier to filter out non-skill extractions (contract terms, location info) before the mapping step. This will reduce the `invalid_extraction` rate and improve evaluation coverage.
2. **Add a confidence threshold:** Set a minimum confidence cutoff (e.g., 0.5) below which the system surfaces a "no confident match" signal rather than a potentially misleading suggestion.
3. **Human-annotated ground truth:** Replace LLM-only annotations with at least 200-300 human-validated items to establish a reliable benchmark.

### Medium-Term
4. **Fine-tune embeddings on job ad + ESCO domain data:** Train a sentence transformer on (extracted_skill, esco_label) pairs for job-specific embeddings.
5. **Leverage ESCO hierarchy:** Take the ESCO skill tree into account when scoring predictions so that close-but-not-exact matches get partial credit, reflecting semantic closeness.
6. **Incorporate job context:** Embed the surrounding sentence or section (not just the extracted skill token) to improve disambiguation for ambiguous terms (e.g., "Python" in programming vs. other contexts).

### Long-Term
7. **Multi-region and multilingual deployment:** Integrate multilingual embeddings (XLM-R, LaBSE) and regional taxonomy variants (i.e O*NET for US).
8. **Ensemble approach:** Use an ensemble of fuzzy matching, fine-tuned embeddings, and a small LLM to pick the best match for high-priority or ambiguous skills, balancing speed and accuracy.